<b>Equipment Replacement as Shortest Path</b>

In [25]:
# Data
Years = range(1,7)
FixedCost = 10000
# The key of the operating cost represents the duration (in years):
OperatingCost = { 1 : 5000, 2 : 12000, 3 : 19000 }

# Define node supply
Supply = {}
for i in Years:
    if i==1:
        Supply[i] = -1
    elif i==6:
        Supply[i] = 1
    else:
        Supply[i] = 0

# Define an arc from year i to year j only if j is between i+1 and i+3 years.
# In other words, the duration j-i must be between 1 and 3:
#
# Option 1:
# Arcs = { (i,j) for i in Years for j in Years if j >= i + 1 if j <= i+3 }
#
# Option 2:
Arcs = { (i,j) for i in Years for j in Years if 1 <= j - i <= 3 }

# Define the cost for each arc: fixed cost + operating cost (depending on duration)
ArcCost = {}
for (i,j) in Arcs:
    ArcCost[i,j] = FixedCost + OperatingCost[j-i]


In [26]:
Arcs

{(1, 2),
 (1, 3),
 (1, 4),
 (2, 3),
 (2, 4),
 (2, 5),
 (3, 4),
 (3, 5),
 (3, 6),
 (4, 5),
 (4, 6),
 (5, 6)}

In [21]:
print(ArcCost)

{(2, 4): 22000, (1, 2): 15000, (3, 4): 15000, (4, 6): 22000, (1, 4): 29000, (2, 3): 15000, (4, 5): 15000, (5, 6): 15000, (3, 6): 29000, (2, 5): 29000, (1, 3): 22000, (3, 5): 22000}


In [22]:
from docplex.mp.model import Model
mdl = Model()

In [23]:
# variables
select = mdl.continuous_var_dict(Arcs, lb=0, name='select')

In [24]:
# objective
mdl.minimize(mdl.sum(ArcCost[i,j]*select[i,j] for (i,j) in Arcs))

In [16]:
# constraints
for j in Years:
    inflow = mdl.sum(select[i,j] for i in Years if (i,j) in Arcs)
    outflow = mdl.sum(select[j,i] for i in Years if (j,i) in Arcs)
    mdl.add_constraint(inflow - outflow == Supply[j])

In [17]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00311303,status='optimal')

In [18]:
mdl.print_solution()

objective: 51000.000
status: OPTIMAL_SOLUTION(2)
  select_4_6=1.000
  select_1_4=1.000


In [19]:
# note that by symmetry of the network, an alternative optimal solution is
#  select_1_3=1.000
#  select_3_6=1.000